# Domain 2: Tool Design & MCP Integration
## TS 2.1: Design Effective Tool Interfaces with Clear Descriptions and Boundaries
**Certification Focus**: CCAR-F Domain 2 (18% Exam Weight) | **Scenario Alignment**: Scenario 1 (Customer Support Resolution Agent)

---

### 🎯 Key Architectural Principles
1. **Tool Descriptions as Primary Routers**: Claude relies on tool descriptions to decide which tool to call. Ambiguous descriptions cause misrouting between overlapping tools.
2. **Explicit Negative Boundaries**: When two tools have overlapping domains (e.g. `get_customer` vs `lookup_order`), explicitly include `BOUNDARY:` instructions specifying when **NOT** to call the tool.
3. **Parameter Constraints & Formatting**: Specify exact string formats (e.g. `'CUST-XXXX'`, `'ORD-XXXX'`) and provide concrete example queries.
4. **Forced Tool Choice**: Using `tool_choice: {"type": "tool", "name": "get_customer"}` to guarantee customer identity verification precedes any financial lookups.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.1 Tool Interface Design.")

### 🛠️ Step 2: Differentiated Tool Declarations with Boundaries
Notice the explicit `BOUNDARY:` directives in both tool descriptions to eliminate misrouting.

In [ ]:
# Step 2: Production Tool Schemas
TOOLS = [
    {
        "name": "get_customer",
        "description": (
            "Retrieves customer identity, contact information, and account status by email address or customer ID. "
            "Use this tool EXCLUSIVELY for verifying user identity, checking membership tier, or retrieving account profiles. "
            "BOUNDARY: Do NOT call this tool for order tracking, tracking numbers, or shipment history; use lookup_order instead. "
            "Example queries: 'Verify customer alice@example.com', 'Retrieve account profile for CUST-101'."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "identifier": {
                    "type": "string",
                    "description": "Customer email (e.g. 'user@example.com') or Customer ID matching format 'CUST-XXXX'.",
                }
            },
            "required": ["identifier"],
        },
    },
    {
        "name": "lookup_order",
        "description": (
            "Retrieves order status, line items, shipment carrier tracking, and return eligibility for a specific order. "
            "Use this tool when users ask about delivery status, order contents, package location, or return windows. "
            "BOUNDARY: Do NOT call this tool for general user profile lookups without an order reference; use get_customer instead. "
            "Example queries: 'Where is order ORD-9821?', 'Check return eligibility for ORD-4412'."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "Order identifier matching format 'ORD-XXXX'.",
                }
            },
            "required": ["order_id"],
        },
    },
]

# Mock Database
CUSTOMER_DB = {"alice@example.com": {"customer_id": "CUST-101", "name": "Alice Smith", "tier": "Gold"}}
ORDER_DB = {"ORD-9821": {"order_id": "ORD-9821", "customer_id": "CUST-101", "carrier": "FedEx", "status": "In Transit"}}

def dispatch_tool(tool_name: str, inputs: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name == "get_customer":
        ident = inputs.get("identifier", "").strip().lower()
        cust = CUSTOMER_DB.get(ident)
        return {"found": True, "customer": cust} if cust else {"found": False, "isError": True, "errorCategory": "validation"}
    elif tool_name == "lookup_order":
        oid = inputs.get("order_id", "").strip().upper()
        ord_rec = ORDER_DB.get(oid)
        return {"found": True, "order": ord_rec} if ord_rec else {"found": False, "isError": True, "errorCategory": "validation"}
    return {"isError": True, "message": f"Unknown tool: {tool_name}"}

print(" Tool schemas and mock dispatch engine ready.")

### 🧪 Step 3: Interactive Simulation
We demonstrate how negative boundaries and forced tool choices direct execution cleanly.

In [ ]:
# Step 3: Simulation
# Test 1: Customer identity retrieval
res_cust = dispatch_tool("get_customer", {"identifier": "alice@example.com"})
print("Customer Lookup Result:", res_cust)

# Test 2: Order tracking retrieval
res_order = dispatch_tool("lookup_order", {"order_id": "ORD-9821"})
print("Order Lookup Result:", res_order)

# Test 3: Forced tool choice representation
forced_choice = {"type": "tool", "name": "get_customer"}
print("\nForced Tool Choice configuration payload:", forced_choice)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Both tool descriptions contain explicit negative boundary instructions.
2. Parameter descriptions specify regex/format conventions.
3. Both tool lookups retrieve accurate data.

In [ ]:
# Step 4: Verification Checks
get_cust_desc = next(t["description"] for t in TOOLS if t["name"] == "get_customer")
lookup_ord_desc = next(t["description"] for t in TOOLS if t["name"] == "lookup_order")

assert "BOUNDARY: Do NOT call this tool for order tracking" in get_cust_desc
assert "BOUNDARY: Do NOT call this tool for general user profile lookups" in lookup_ord_desc
assert "CUST-XXXX" in get_cust_desc or "CUST-XXXX" in str(TOOLS)
assert "ORD-XXXX" in lookup_ord_desc or "ORD-XXXX" in str(TOOLS)

assert res_cust["found"] is True
assert res_cust["customer"]["customer_id"] == "CUST-101"
assert res_order["order"]["carrier"] == "FedEx"

print(" All verification assertions passed! Tool interfaces verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Tool descriptions must define explicit **negative boundaries** (`BOUNDARY: Do NOT call this tool when...`).
- **Rule 2**: Specify exact parameter format constraints (`CUST-XXXX`) and provide concrete query examples.
- **Rule 3**: Use forced `tool_choice` (`{"type": "tool", "name": "..."}`) when a step is non-negotiable.